# MarketForge AI — Exploratory Data Analysis

Use this notebook to explore the ingested job data after your first pipeline run.

```bash
# Run from project root:
jupyter notebook notebooks/
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from marketforge.memory.postgres import get_sync_engine
from sqlalchemy import text

engine = get_sync_engine()

# Load jobs
with engine.connect() as conn:
    df = pd.read_sql('SELECT * FROM market.jobs ORDER BY scraped_at DESC LIMIT 1000', conn)

print(f'Loaded {len(df)} jobs')
df.head()

In [ ]:
# Salary distribution
sal_df = df.dropna(subset=['salary_min', 'salary_max'])
sal_df['midpoint'] = (sal_df['salary_min'] + sal_df['salary_max']) / 2
sal_df['midpoint'].hist(bins=30, figsize=(10,4))
plt.title('Salary Midpoint Distribution')
plt.xlabel('Salary (£)')
plt.show()

In [ ]:
# Top skills
with engine.connect() as conn:
    skills_df = pd.read_sql(
        'SELECT skill, COUNT(*) as count FROM market.job_skills GROUP BY skill ORDER BY count DESC LIMIT 20',
        conn
    )
skills_df.plot.barh(x='skill', y='count', figsize=(10,6))
plt.title('Top 20 Skills')
plt.tight_layout()
plt.show()